# Bronze Layer — Raw Data Ingestion

**Medallion Stage:** Bronze (Raw)

This notebook covers the first stage of the Medallion architecture:
loading the raw IBM HR Attrition dataset, profiling it without modifications,
and persisting it to the bronze layer with a metadata manifest.

> **No transformations happen here.** Bronze = raw, immutable truth.

In [1]:
import sys
from pathlib import Path

# Ensure project root is on path so src/ imports work
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
import json
from IPython.display import display

# Plot aesthetics
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})
sns.set_palette('Set2')
print(f'Working root: {ROOT}')

Working root: C:\Documents\GitHubRepos\PowerBIProject\Employee Attrition


## 1. Load Raw CSV

Place `WA_Fn-UseC_-HR-Employee-Attrition.csv` into `data/bronze/` before running.
Download: https://www.kaggle.com/datasets/patelprashant/employee-attrition

In [2]:
RAW_CSV = ROOT / 'data' / 'bronze' / 'WA_Fn-UseC_-HR-Employee-Attrition.csv'

if not RAW_CSV.exists():
    raise FileNotFoundError(
        f'Raw CSV not found at {RAW_CSV}\n'
        'Download from Kaggle and place it at data/bronze/WA_Fn-UseC_-HR-Employee-Attrition.csv'
    )

df = pd.read_csv(RAW_CSV)
print(f'Shape: {df.shape[0]:,} rows  x  {df.shape[1]} columns')

Shape: 1,470 rows  x  35 columns


## 2. Schema Overview

In [3]:
dtype_df = pd.DataFrame({
    'Column': df.columns,
    'Dtype': df.dtypes.values,
    'Non-Null': df.notna().sum().values,
    'Null Count': df.isnull().sum().values,
    'Null %': (df.isnull().mean() * 100).round(2).values,
    'Unique Values': df.nunique().values,
    'Sample': [df[c].dropna().iloc[0] if df[c].notna().any() else 'N/A' for c in df.columns],
})
display(dtype_df)

,Column,Dtype,Non-Null,Null Count,Null %,Unique Values,Sample
0,Age,int64,1470,0,0.0,43,41
1,Attrition,str,1470,0,0.0,2,Yes
2,BusinessTravel,str,1470,0,0.0,3,Travel_Rarely
3,DailyRate,int64,1470,0,0.0,886,1102
4,Department,str,1470,0,0.0,3,Sales
5,DistanceFromHome,int64,1470,0,0.0,29,1
6,Education,int64,1470,0,0.0,5,2
7,EducationField,str,1470,0,0.0,6,Life Sciences
8,EmployeeCount,int64,1470,0,0.0,1,1
9,EmployeeNumber,int64,1470,0,0.0,1470,1


## 3. Missingness Map

This dataset is known to be complete — the matrix below confirms it.

In [4]:
fig, ax = plt.subplots(figsize=(14, 4))
msno.bar(df, ax=ax, color='steelblue', fontsize=9)
ax.set_title('Completeness per Column', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

null_total = df.isnull().sum().sum()
print(f'Total null cells: {null_total}  ({"No missing data" if null_total == 0 else "MISSING DATA FOUND"})')

Total null cells: 0  (No missing data)


C:\Users\ewach\AppData\Local\Temp\ipykernel_27092\2386906574.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Constant Columns

Three columns carry the same value for every row — they are artefacts of the dataset
generation and will be removed in the Silver layer.

In [5]:
constants = [col for col in df.columns if df[col].nunique() == 1]
print(f'Constant columns ({len(constants)}):')
for col in constants:
    print(f'  {col:30s} = {df[col].iloc[0]!r}')

Constant columns (3):
  EmployeeCount                  = np.int64(1)
  Over18                         = 'Y'
  StandardHours                  = np.int64(80)


## 5. Duplicate Check

In [6]:
dup_rows = df.duplicated().sum()
dup_emp  = df.duplicated(subset=['EmployeeNumber']).sum()
print(f'Duplicate full rows       : {dup_rows}')
print(f'Duplicate EmployeeNumbers : {dup_emp}')
assert dup_emp == 0, 'Duplicate employee IDs detected — investigate before proceeding'

Duplicate full rows       : 0
Duplicate EmployeeNumbers : 0


## 6. Target Variable Distribution

In [7]:
counts = df['Attrition'].value_counts()
rate   = counts['Yes'] / len(df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='white')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Attrition Count', fontweight='bold')
axes[0].set_ylabel('Employees')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Attrition Proportion', fontweight='bold')

plt.suptitle(f'Overall Attrition Rate: {rate:.1%}', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Attrition rate: {rate:.1%}  |  Class imbalance ratio: 1 : {(1-rate)/rate:.1f}')

Attrition rate: 16.1%  |  Class imbalance ratio: 1 : 5.2


C:\Users\ewach\AppData\Local\Temp\ipykernel_27092\3248475643.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Numerical Feature Summary

In [8]:
num_cols = df.select_dtypes(include='number').columns.tolist()
desc = df[num_cols].describe().T
desc['skew'] = df[num_cols].skew().round(3)
desc['kurt'] = df[num_cols].kurtosis().round(3)
display(desc.style.background_gradient(subset=['mean', 'std', 'skew'], cmap='RdYlGn_r'))

,count,mean,std,min,25%,50%,75%,max,skew,kurt
Age,1470.000000,36.923810,9.135373,18.000000,30.000000,36.000000,43.000000,60.000000,0.413000,-0.404000
DailyRate,1470.000000,802.485714,403.509100,102.000000,465.000000,802.000000,1157.000000,1499.000000,-0.004000,-1.204000
DistanceFromHome,1470.000000,9.192517,8.106864,1.000000,2.000000,7.000000,14.000000,29.000000,0.958000,-0.225000
Education,1470.000000,2.912925,1.024165,1.000000,2.000000,3.000000,4.000000,5.000000,-0.290000,-0.559000
EmployeeCount,1470.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000
EmployeeNumber,1470.000000,1024.865306,602.024335,1.000000,491.250000,1020.500000,1555.750000,2068.000000,0.017000,-1.223000
EnvironmentSatisfaction,1470.000000,2.721769,1.093082,1.000000,2.000000,3.000000,4.000000,4.000000,-0.322000,-1.203000
HourlyRate,1470.000000,65.891156,20.329428,30.000000,48.000000,66.000000,83.750000,100.000000,-0.032000,-1.196000
JobInvolvement,1470.000000,2.729932,0.711561,1.000000,2.000000,3.000000,3.000000,4.000000,-0.498000,0.271000
JobLevel,1470.000000,2.063946,1.106940,1.000000,1.000000,2.000000,3.000000,5.000000,1.025000,0.399000


## 8. Categorical Value Counts

In [9]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols.remove('Attrition')  # already covered above

n_cols = 3
n_rows = -(-len(cat_cols) // n_cols)  # ceiling division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    vc = df[col].value_counts()
    axes[i].barh(vc.index, vc.values, color=sns.color_palette('Set2', len(vc)))
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Count')
    for j, v in enumerate(vc.values):
        axes[i].text(v + 5, j, str(v), va='center', fontsize=9)

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

C:\Users\ewach\AppData\Local\Temp\ipykernel_27092\3288277146.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


C:\Users\ewach\AppData\Local\Temp\ipykernel_27092\3288277146.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Save Bronze Layer

In [10]:
from src.utils.config import load_config
from src.bronze.ingest import BronzeIngester

cfg = load_config()
ingester = BronzeIngester(cfg)
bronze_df = ingester.ingest(RAW_CSV)

# Verify metadata
meta_path = ROOT / 'data' / 'bronze' / 'metadata.json'
with open(meta_path) as f:
    meta = json.load(f)

print('Bronze metadata saved:')
for k, v in meta.items():
    if k not in ('columns', 'dtypes', 'null_counts'):
        print(f'  {k}: {v}')

2026-05-02 11:38:08 | INFO     | src.bronze.ingest | Ingesting raw data from: C:\Documents\GitHubRepos\PowerBIProject\Employee Attrition\data\bronze\WA_Fn-UseC_-HR-Employee-Attrition.csv


2026-05-02 11:38:08 | INFO     | src.bronze.ingest | Loaded 1,470 rows x 35 columns


2026-05-02 11:38:08 | INFO     | src.bronze.ingest | Bronze validation passed


2026-05-02 11:38:08 | INFO     | src.bronze.ingest | Bronze layer written: C:\Documents\GitHubRepos\PowerBIProject\Employee Attrition\data\bronze\WA_Fn-UseC_-HR-Employee-Attrition.csv


2026-05-02 11:38:08 | INFO     | src.bronze.ingest | Ingestion metadata written: C:\Documents\GitHubRepos\PowerBIProject\Employee Attrition\data\bronze\metadata.json


Bronze metadata saved:
  ingestion_timestamp: 2026-05-02T15:38:08.848810+00:00
  source_file: C:\Documents\GitHubRepos\PowerBIProject\Employee Attrition\data\bronze\WA_Fn-UseC_-HR-Employee-Attrition.csv
  row_count: 1470
  column_count: 35
  attrition_distribution: {'No': 1233, 'Yes': 237}


---
## Summary

| Item | Value |
|---|---|
| Rows | 1,470 |
| Columns | 35 |
| Missing values | 0 |
| Constant columns | 3 (EmployeeCount, Over18, StandardHours) |
| Attrition rate | ~16.1% |
| Class imbalance | 1 : 5.2 |

**Next step:** `02_silver_data_cleaning.ipynb` — clean, encode, and enrich.